In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from utils import agop_gcnn as agc
from utils import trainer as tr
from torch.utils.data import Dataset
import random
import torch.backends.cudnn as cudnn
#import rfm
import numpy as np
from trained_models.CIFAR.model3 import trainer as t
import numpy as np
from sklearn.model_selection import train_test_split
from torch.linalg import norm
from torchvision import models
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn
from utils.groupy.gconv.pytorch_gconv.splitgconv2d import P4ConvZ2, P4ConvP4, P4MConvZ2, P4MConvP4M
from groupy.gconv.make_gconv_indices import *
from copy import deepcopy
from torch.nn.functional import pad
from torch.func import jacrev

Setting up a new session...
Without the incoming socket you cannot receive events from the server or register event handlers to your Visdom client.


In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
#device='cpu'
print(f"Using device: {device}")


Using device: cuda:0


In [3]:
!nvidia-smi

Sun Sep 21 22:09:24 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.04             Driver Version: 570.124.04     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:05:00.0 Off |                  Off |
|  0%   43C    P8             18W /  450W |       4MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
torch.cuda.empty_cache()

In [ ]:
def agop_trajectory(epoch, kwargs):
     corr_list=[]
     for i in range(14):
         G,corr = agc.verify_NFA(kwargs['net'], kwargs['init_net'], kwargs['train_loader'] , layer_idx=i, chunk_size=2, max_batches=10, centering=False)
         corr_list.append(corr)
     return corr_list

In [4]:
pwd = os.getcwd()+'/trajectory.pt'
trainloader, valloader, testloader, init_net, net = t.train_net(force_train=False, fn=agop_trajectory, kwargs={'save_path': pwd})

Files already downloaded and verified
Files already downloaded and verified
Model weights loaded successfully.


In [7]:
import torchvision.transforms as transforms
train_dataset = trainloader.dataset
means = (0.4914, 0.4822, 0.4465)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(means, (0.2023, 0.1994, 0.2010))
])
trainloader1 = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2
)

In [15]:
#tr.visualize_predictions(net, testloader, range(10), device, num_images=36)

print(tr.get_acc_ce(net.to(device), trainloader1))

99.73


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
matplotlib.use('Agg')
def plot_correlation_trajectory_from_tensors(data):

    # Sort epochs to ensure chronological order
    epochs = sorted(data.keys())

    # Extract correlations: For each epoch, extract the scalar value (.item()) 
    # from each PyTorch Tensor in the list.
    # FIX: Use .cpu().item() to safely move the tensor to host memory before conversion.
    correlations = [[t.cpu().item() for t in data[e]] for e in epochs]
    
    # Transpose to get correlations grouped by layer (layer-first view)
    correlations_np = np.array(correlations).T
    num_layers = correlations_np.shape[0]

    # Use a distinct, colorblind-friendly color map (e.g., 'Dark2')
    # Use plt.cm.get_cmap which is a reliable way to get an indexed color list
    colors = plt.cm.get_cmap('Dark2', num_layers).colors
    
    plt.figure(figsize=(10, 6))
    
    for i in range(num_layers):
        # Plot correlations for the i-th layer, using a unique color
        plt.plot(epochs, correlations_np[i], 
                 marker='o', 
                 linestyle='-', 
                 markersize=6, 
                 color=colors[i],
                 label=f'Layer {i+1}')

    plt.title('Layer Correlation Trajectory Across Training Epochs', fontsize=16)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Correlation (Pearson)', fontsize=12)
    
    # Ensure all epoch ticks are visible
    plt.xticks(epochs) 
    
    plt.legend(title='Convolutional Layers', loc='best', frameon=True)
    plt.grid(True, linestyle='--', alpha=0.6)
    
    # Adjust y-limit dynamically to zoom in on the data
    min_corr = np.min(correlations_np)
    max_corr = np.max(correlations_np)
    # Set y-limits slightly outside the data range
    plt.ylim(min_corr - 0.005, max_corr + 0.005)
    
    plt.tight_layout()

    # Save the plot
    output_filename = 'layer_correlation_trajectory_non_gui.png'
    plt.savefig(output_filename)
    print(f"Plot saved to {output_filename}")


data= torch.load('trajectory.pt', weights_only= True)
plot_correlation_trajectory_from_tensors(data)